<a href="https://colab.research.google.com/github/Freibott/clasificador-milito/blob/main/clasificador_milito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificador de textos en esperanto: Segunda Guerra Mundial, Guerra Civil Española y Guerras Napoleónicas

Minería de Textos · CUNEF · 2026/2027

## Fase 1 · Dataset

### Exploración de fuentes

**Objetivo.** Antes de construir el dataset comprobamos (30/10/2026) si Vikipedio (Wikipedia en esperanto) tiene texto suficiente para las tres clases. Para ello consultamos su API de MediaWiki en modo de solo lectura. La bitácora completa está en `docs/bitacora_exploracion_fase1.md` del repositorio.

**Por qué estas clases.** Empezamos con WWI, WWII y Guerra Fría, pero a profundidad 1 compartían 32 artículos (sobre todo en 1943–1949 y en el periodo de entreguerras). Eso mete ruido en las etiquetas y limita la precisión. Las sustituimos por tres guerras del mismo tema y con muy pocos artículos en común. Antes descartamos otras candidatas por falta de datos: Vietnam, Afganistán, Aceh, la guerra civil china y Corea.

**Hallazgos (páginas directas, recuento completo).**

| | Segunda Guerra Mundial | Guerra Civil Española | Guerras Napoleónicas |
|---|---|---|---|
| Categoría | `Kategorio:Dua Mondmilito` | `Kategorio:Hispana Enlanda Milito` | `Kategorio:Napoleonaj militoj` |
| Artículos directos | 132 | 54 | 41 |
| Subcategorías | 27 | 4 | 8 |
| Palabras mín / mediana / máx | 12 / 284 / 12.238 | 58 / 386 / 5.050 | 76 / 279 / 1.780 |
| Párrafos (≥10 palabras) por artículo | 25,1 | 14,9 | 7,3 |
| Párrafos en páginas directas | 3.312 | 802 | 298 |

**Solapes entre páginas directas:** ninguno.

**Valoración.** Es viable. Las clases son del mismo tema, como pide el enunciado. La Guerra Civil Española y la Segunda Guerra Mundial son cercanas (1936–1945, fascismo, intervención alemana e italiana), así que el problema no es trivial. Con solo las páginas directas, las Guerras Napoleónicas no llegan a 300 párrafos, por lo que estudiamos las subcategorías.

**Incidencias resueltas.** Los nombres de categoría distinguen mayúsculas. La API de extractos solo devuelve un texto completo por petición. Además, hubo que gestionar el error 429 (demasiadas peticiones) con reintentos y pausas.

### Estudio de profundidad de subcategorías

**Objetivo.** Medimos (30/10/2026) cuántos párrafos únicos se obtienen al bajar 0, 1 o 2 niveles en el árbol de categorías de cada clase. Se descargaron los 1.306 artículos distintos hasta profundidad 2, así que es un recuento completo. El detalle está en `docs/estudio_profundidad_fase1.md` del repositorio.

**Párrafos únicos por clase** (líneas de 10 palabras o más, sin contar los artículos que comparten varias clases):

| Profundidad | Segunda Guerra Mundial | Guerra Civil Española | Guerras Napoleónicas |
|---|---:|---:|---:|
| 0 (páginas directas) | 3.312 | 802 | 298 |
| 1 | 5.371 | 1.689 | 735 |
| 2 | 11.880 | 2.034 | 746 |

**Solapes.** A profundidad 1 solo hay 2 artículos compartidos, ambos entre la Segunda Guerra Mundial y la Guerra Civil Española: Hugo Sperrle e Hispana ekonomia miraklo. A profundidad 2 son 9. Las Guerras Napoleónicas no comparten ninguno.

**Decisiones.**
1. **Profundidad 1.** Es la mínima con la que las Guerras Napoleónicas superan los 300 párrafos (735). La profundidad 2 apenas les aporta (+11) y duplica la Segunda Guerra Mundial.
2. **Excluir:**
   - "Sekvoj de la Dua Mondmilito" y "Frankismo", porque son de posguerra y por ellas entra el solape "Hispana ekonomia miraklo".
   - "Brit-usona milito de 1812", porque es otra guerra.
   - El artículo "Listo de svedoj kiuj mortis en la Hispana Enlanda Milito", porque es una lista de nombres.

   Las subcategorías dudosas (biografías, lugares, obras) se mantienen y su ruido se mide más adelante.
3. **Descartar los artículos presentes en más de una clase.** Tras las exclusiones solo queda uno (Hugo Sperrle).
4. **Equilibrar las clases** recortando cada una al tamaño de la menor (Guerras Napoleónicas, unos 720 párrafos), con muestreo aleatorio y semilla fija. En la Fase 3 se compara con el dataset sin equilibrar.
5. **Conservar números y fechas** por ahora. En la Fase 3 se entrena con y sin ellos, porque los años (1799–1815, 1936–1939, 1939–1945) pueden delatar la clase casi solos.

### Generación del dataset

El código de generación está en el cuaderno, como pide el enunciado, pero se puede desactivar con `GENERAR_DATASET`. Con `False`, el cuaderno carga el CSV ya generado desde el repositorio. Con `True`, lo vuelve a construir desde la API de Vikipedio, lo que tarda varios minutos por las pausas entre peticiones.

In [1]:
# --- Importaciones ---
import re
import time
import random
from collections import deque, Counter

import requests
import pandas as pd

# --- Parámetros de ejecución ---
GENERAR_DATASET = True   # True: construir el dataset desde la API · False: cargar el CSV del repo

# --- Fuente y categorías (decisiones del estudio de profundidad) ---
API_URL = "https://eo.wikipedia.org/w/api.php"
CATEGORIAS = {                                   # clase -> categoría raíz (distingue mayúsculas)
    "Segunda Guerra Mundial": "Kategorio:Dua Mondmilito",
    "Guerra Civil Española": "Kategorio:Hispana Enlanda Milito",
    "Guerras Napoleónicas": "Kategorio:Napoleonaj militoj",
}
PROFUNDIDAD = 1                                  # decisión 1
SUBCATS_EXCLUIDAS = {                            # decisión 2
    "Kategorio:Sekvoj de la Dua Mondmilito",     # posguerra
    "Kategorio:Frankismo",                       # posguerra (dictadura 1939-1975)
    "Kategorio:Brit-usona milito de 1812",       # otra guerra
}
ARTICULOS_EXCLUIDOS = {                          # decisión 2
    "Listo de svedoj kiuj mortis en la Hispana Enlanda Milito",  # lista de nombres
}

# --- Limpieza y equilibrado ---
MIN_PALABRAS = 10                                # párrafo = línea con 10 palabras o más
SEMILLA = 42                                     # decisión 4 (reproducibilidad)

# --- Cortesía con la API (evita el error 429) ---
PAUSA = 1.0                                      # segundos entre peticiones
MAX_REINTENTOS = 4
USER_AGENT = "ClasificadorMilito/0.1 (https://github.com/Freibott/clasificador-milito; práctica CUNEF)"

# --- Rutas ---
RUTA_CSV = "data/dataset.csv"
URL_CSV_REMOTO = "https://raw.githubusercontent.com/Freibott/clasificador-milito/main/data/dataset.csv"

In [2]:
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})  # Wikimedia exige identificarse


def consultar_api(params):
    """Hace una petición GET a la API de MediaWiki y devuelve el JSON como diccionario.

    Centraliza la identificación (User-Agent), las pausas entre peticiones y los
    reintentos con espera creciente ante el límite de peticiones (error 429).
    """
    p = dict(params)                 # copia: no modificamos el diccionario de quien llama
    # Establecemos formato json
    p["format"] = "json"
    p["formatversion"] = 2           # MediaWiki: devuelve listas en vez de diccionarios por ID

    for intento in range(MAX_REINTENTOS):
        respuesta = session.get(API_URL, params=p, timeout=30)

        # 429 -> demasiadas peticiones, 5xx -> error del servidor: se reintenta
        if respuesta.status_code == 429 or respuesta.status_code >= 500:
            time.sleep(PAUSA * 2**intento)   # espera creciente (1, 2, 4, 8 s)
            continue

        respuesta.raise_for_status()         # otros errores HTTP (400, 403, 404): no se reintentan
        datos = respuesta.json()

        # MediaWiki responde 200 incluso con errores de la consulta
        if "error" in datos:
            raise RuntimeError(f"Error en la API de MediaWiki: {datos['error']['info']}")

        time.sleep(PAUSA)                    # pausa para no saturar el servidor
        return datos

    # Solo se llega aquí si se agotan todos los intentos
    raise RuntimeError(f"La API falló tras {MAX_REINTENTOS} intentos, último código: {respuesta.status_code}")

Comprobación: la raíz de la Segunda Guerra Mundial debe devolver 161 miembros directos (132 artículos, 27 subcategorías y 2 páginas de otros espacios de nombres).

In [3]:
datos = consultar_api({
    "action": "query",
    "list": "categorymembers",
    "cmtitle": CATEGORIAS["Segunda Guerra Mundial"],
    "cmlimit": 500,
})
len(datos["query"]["categorymembers"])  # → 161

161

### Bloque 2 · Artículos de cada clase

En este bloque se recopilan todos los artículos que componen cada una de las tres clases temáticas consultando la API de Wikipedia. Se utiliza una profundidad de 1 para obtener los artículos de la categoría raíz y los de sus subcategorías directas más relevantes. Este límite acota la búsqueda y evita el ruido temático o la dispersión excesiva que provocaría descender a niveles más profundos en la jerarquía.

In [4]:
def miembros_de_categoria(titulo, limite=500):
    """Devuelve (paginas, subcats): títulos de artículos (ns 0) y subcategorías (ns 14)."""
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": titulo,
        "cmlimit": limite,
        "cmtype": "page|subcat",  # ignora ficheros e imágenes (ns 6)
    }
    paginas, subcats = [], []
    while True:
        datos = consultar_api(params)
        for m in datos["query"]["categorymembers"]:
            if m["ns"] == 0:
                paginas.append(m["title"])
            elif m["ns"] == 14:
                subcats.append(m["title"])
        if "continue" not in datos:
            break
        params.update(datos["continue"])
    return paginas, subcats

In [5]:
for lim in (500, 20):
    p, s = miembros_de_categoria(CATEGORIAS["Segunda Guerra Mundial"], limite=lim)
    print(lim, len(p), len(s), len(set(p)))

for clase, raiz in CATEGORIAS.items():
    p, s = miembros_de_categoria(raiz)
    print(clase, len(p), len(s), s)

500 132 27 132
20 132 27 132
Segunda Guerra Mundial 132 27 ['Kategorio:Dua Mondmilito laŭ landoj', 'Kategorio:Kronologio de la Dua Mondmilito', 'Kategorio:Dua Mondmilito laŭ kontinentoj', 'Kategorio:Bataloj kaj operacoj de la Dua Mondmilito', 'Kategorio:Bestoj en la Dua Mondmilito', 'Kategorio:Detencentroj de la Dua Mondmilito', 'Kategorio:Historiografio de la Dua Mondmilito', 'Kategorio:Koncentrejo', 'Kategorio:Naziaj koncentrejoj', 'Kategorio:Kultura historio de la Dua Mondmilito', 'Kategorio:Kunlaborado kun nazioj', 'Kategorio:Lokoj ligitaj al la Dua Mondmilito', 'Kategorio:Masakroj en la Dua Mondmilito', 'Kategorio:Memoro de la Dua Mondmilito', 'Kategorio:Milita ekipaĵo de la Dua Mondmilito', 'Kategorio:Militaj premioj de la Dua Mondmilito', 'Kategorio:Militkrimoj dum la Dua Mondmilito', 'Kategorio:Monumentoj de la Dua Mondmilito', 'Kategorio:Movadoj de Rezistado en la Dua Mondmilito', 'Kategorio:Okupitaj teritorioj de la Dua Mondmilito', 'Kategorio:Personoj de la Dua Mondmilito', 

In [6]:
def recorrer_categoria(raiz, profundidad=PROFUNDIDAD, excluidas=SUBCATS_EXCLUIDAS):
    """Recorre el árbol de categorías en anchura (BFS) y devuelve {artículo: categoría_origen}."""
    cola = deque([(raiz, 0)])
    visitadas = {raiz}
    articulos = {}

    while cola:
        cat, nivel = cola.popleft()
        pags, subcats = miembros_de_categoria(cat)

        for pagina in pags:
            articulos.setdefault(pagina, cat)

        if nivel < profundidad:
            for s in subcats:
                if s not in excluidas and s not in visitadas:
                    visitadas.add(s)
                    cola.append((s, nivel + 1))

    return articulos

In [7]:
sin_excl = {c: recorrer_categoria(r, excluidas=set()) for c, r in CATEGORIAS.items()}
articulos_por_clase = {c: recorrer_categoria(r) for c, r in CATEGORIAS.items()}  # la usará el Bloque 3

for c in CATEGORIAS:
    print(f"{c}: {len(sin_excl[c])} sin excl. · {len(articulos_por_clase[c])} con excl.")

for nombre, d in (("sin", sin_excl), ("con", articulos_por_clase)):
    cuenta = Counter(t for arts in d.values() for t in arts)
    print(nombre, "solapes:", [t for t, n in cuenta.items() if n > 1])

Segunda Guerra Mundial: 343 sin excl. · 326 con excl.
Guerra Civil Española: 150 sin excl. · 134 con excl.
Guerras Napoleónicas: 97 sin excl. · 95 con excl.
sin solapes: ['Hugo Sperrle', 'Hispana ekonomia miraklo']
con solapes: ['Hugo Sperrle']


### Bloque 3 · Descarga de texto

In [8]:
import os, json, random
from tqdm.auto import tqdm

def descargar_texto(titulo):
    params = {"action": "query", "prop": "extracts", "explaintext": 1,
              "exsectionformat": "wiki", "redirects": 1, "titles": titulo,
              "format": "json", "formatversion": 2}
    for _ in range(3):
        try:
            r = session.get(API_URL, params=params, timeout=30)
            r.raise_for_status()
            return r.json()["query"]["pages"][0].get("extract")  # None si falta
        except (requests.RequestException, ValueError, KeyError):
            time.sleep(2)
    return None

t = descargar_texto("Napoleono Bonaparte")
print(type(t), len(t), "==" in t)   # str, varios miles, True

<class 'str'> 77634 True


In [9]:
titulos = sorted(set().union(*articulos_por_clase.values()) - ARTICULOS_EXCLUIDOS)
print(len(titulos))   # → 553

553


In [10]:
RUTA_TEXTOS = "data/raw/textos.json"
os.makedirs("data/raw", exist_ok=True)

if os.path.exists(RUTA_TEXTOS):
    with open(RUTA_TEXTOS, encoding="utf-8") as f:
        textos = json.load(f)
else:
    textos = {}
    for t in tqdm(titulos):
        textos[t] = descargar_texto(t)
        time.sleep(0.2)
    with open(RUTA_TEXTOS, "w", encoding="utf-8") as f:
        json.dump(textos, f, ensure_ascii=False)

  0%|          | 0/553 [00:00<?, ?it/s]

In [11]:
print(len(textos))                                    # → 553
nones = [t for t, v in textos.items() if v is None]
cortos = [t for t, v in textos.items() if v is not None and len(v) < 200]
print(len(nones), nones)                              # idealmente 0
print(len(cortos), cortos)

random.seed(42)
for t in random.sample(sorted(textos), 2):
    print(t, "→", (textos[t] or "")[:300], "\n")

553
0 []
2 ['Kvanto de viktimoj de la dua mondmilito', 'La kanto de la samideanoj']
Ekstermejo → Ekstermejo (germane: Vernichtungslager) - ankaŭ nomata mortkoncentrejo (Todeslager) aŭ pereigejo - estas tiu speco de koncentrejo, kie oni celas tuj mortigi la tie alveturigitajn homojn. Organizitaj de Germanio tiam nomata Tria Regno dum la Dua Mondmilito. Ili funkciis de decembro 1941 ĝis januaro 1 

Antonín Sochor → Generalmajoro Antonín Sochor estis komandanto de motorizita bataliono de mitralistoj de la 1-a ĉeĥoslovaka memstara brigado en Sovetunio kaj Heroo de Sovetunio.


== Antaŭ la milito ==
Antonín Sochor naskiĝis la 16-an de junio 1916 en Lohberg en Germanio, kien liaj gepatroj, same kiel centoj da nord 



## Fase 2 · NLP

## Fase 3 · Clasificación